# Wan2.1 14B: bounded port acceptance

**Status: port not yet accepted.** Use **A100 40 GB with high RAM**; model CPU offload is selected automatically. An 80 GB A100/H100 also works without offload. The 40 GB option may run more slowly; the new experiment's total cost is not yet measured. This notebook does not run the older probe suites.

Maximum: **four complete sampling trajectories**: native pipeline, guidance off, intended reference, reversed reference. The first two establish pretrained parity. The latter two reuse the verified off baseline. A single extra CFG forward at sampling index 9 traces propagation from an identical UniPC solver state. No sweeps.

Use the companion `wan_port_acceptance_bundle.zip`; it contains the exact patched source and the 21-frame camel input. No GitHub push is required. Do not substitute a stale checkout.


## 1. Fresh-runtime dependencies

Run once, then use **Runtime → Restart session** before continuing. Keep the Colab CUDA/PyTorch installation; its exact version is recorded. Do not install the repository's CogVideoX requirements.


In [ ]:
%pip install "diffusers==0.39.0" "transformers==4.57.6" "huggingface-hub==0.36.2" "ftfy==6.3.1" "tokenizers==0.22.2" "accelerate==1.14.0" imageio imageio-ffmpeg omegaconf einops


## 2. Load the exact source bundle

Upload `wan_port_acceptance_bundle.zip` when prompted. This extracts into a separate directory.


In [ ]:
from pathlib import Path
import os, zipfile, json
from google.colab import files
PROJECT = Path('/content/ditflow_port_acceptance')
BUNDLE_REVISION = 'wan-port-40gb-v1'
old_manifest = PROJECT / 'bundle_manifest.json'
if not old_manifest.is_file() or json.loads(old_manifest.read_text()).get('revision') != BUNDLE_REVISION:
    uploaded = files.upload()
    bundle = next(Path(name) for name in uploaded if name == 'wan_port_acceptance_bundle.zip')
    PROJECT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(bundle) as archive:
        for item in archive.infolist():
            dest = (PROJECT / item.filename).resolve()
            if not dest.is_relative_to(PROJECT.resolve()):
                raise ValueError('Unsafe archive path')
        archive.extractall(PROJECT)
os.chdir(PROJECT)
print(PROJECT)


## 3. Runtime and source preflight

This does not run model experiments. The weight-free gates were already tested locally; this cell checks runtime identity and bundle integrity.


In [ ]:
import json, hashlib, torch
from importlib.metadata import version
expected = {'diffusers':'0.39.0','transformers':'4.57.6','huggingface-hub':'0.36.2','ftfy':'6.3.1','tokenizers':'0.22.2','accelerate':'1.14.0'}
actual = {name:version(name) for name in expected}
assert actual == expected, {'expected':expected, 'actual':actual}
assert torch.cuda.is_available()
assert torch.cuda.get_device_properties(0).total_memory >= 35*2**30, 'Select A100 40 GB or A100/H100 80 GB, with high RAM'
manifest = json.loads(Path('bundle_manifest.json').read_text())
assert manifest.get('revision') == BUNDLE_REVISION, 'Upload the updated 40 GB bundle'
for name, digest in manifest['sha256'].items():
    assert hashlib.sha256(Path(name).read_bytes()).hexdigest() == digest, name
print(actual, torch.__version__, torch.version.cuda, torch.cuda.get_device_name())


## 4. Pin the checkpoint and establish native parity

**Hypothesis:** guidance-off follows the installed native WanPipeline with the same checkpoint, noise, prompt embeddings, precision policy and solver.

**Expected:** every next latent matches within `atol=rtol=1e-5`. **Maximum:** two complete unguided sampling trajectories. Failure blocks guided generation.

Frozen experiment: T2V 14B; 480×832; 21 RGB / 6 latent frames; 50 UniPC steps, shift 3; CFG 5; seed 1. Image conditioning is **absent** in T2V. AMF later uses block 20, mean-head logits, temperature 2, native RoPE, all ordered pairs, hard reference / soft target, five Adam updates at indices 0–9 with LR 0.002→0.001. KV injection is off. The reference is the existing camel clip. The prompt leaves direction unspecified.


In [ ]:
from benchmark.wan_port_acceptance import AcceptanceRun
run = AcceptanceRun(
    output='probe_runs/wan_port_acceptance_camel_s1',
    video='probe_runs/wan_reference_inputs/clips/camel',
    prompt='A camel walking across a dusty paddock beside a metal fence.',
    seed=1,
)
run.parity()
print(json.dumps(json.loads((run.root/'parity.json').read_text()), indent=2))


## 5. Review the vanilla video before spending on guidance

Look for a coherent camel and visible motion through the clip. Reject a static, badly damaged or incoherent baseline. Do not accept it because tensors differ. The review below is an experimental quality gate requested in the task.


In [ ]:
from IPython.display import display
from colab_utils import show_videos
display(show_videos([run.root/'native/final.mp4',run.root/'off/final.mp4'],['Native pipeline','Port guidance off']))


In [ ]:
VANILLA_REVIEW = {
    'credible_motion': False,  # Set True only after watching the videos above.
    'notes': '',              # Describe observed subject travel/gait and visible defects.
}


## 6. Two reference conditions, one propagation trace

**Hypothesis:** the reference order steers decoded subject motion. **Expected:** intended and reversed references produce distinguishable, appropriately directed subject trajectories. Appearance changes and AMF loss reduction do not pass this test.

**Maximum:** two guided generations and one additional pre-update CFG forward at index 9. All settings are fixed; only the decoded reference-frame order changes between guided arms. RGB reversal happens before VAE encoding. No head/layer/LR sweep follows automatically.


In [ ]:
display(run.compare(VANILLA_REVIEW))
print(json.dumps(json.loads((run.root/'forward/step_trace.json').read_text()), indent=2))


## 7. Save the evidence for independent decoded-motion evaluation

Download the bundle and return it for review. It contains the manifest, exact checkpoint revision/configuration, parity measurements, solver state, before/after latent and prediction arrays, and side-by-side videos. `status.json` deliberately does not claim success.

Acceptance requires tracking the generated subject in decoded RGB, checking foreground versus background/camera movement, and watching the videos. If the first case succeeds, confirm **camel seed 2**, then **a second motion case** with a fresh matched native/off/intended/reversed comparison and the same frozen mechanism. Do not launch confirmations before the first decoded result is assessed.


In [ ]:
import shutil
archive = shutil.make_archive(str(run.root), 'zip', root_dir=run.root)
print(archive)
files.download(archive)
